# Double descent of the QELM on MNIST digits

Issue #181. A static, non-linear classification task, MNIST digits 3 against 5 by default (or more digits, `CLASSES`), on which the generalization error of the QELM readout shows the double descent around the interpolation threshold `alpha = N_train / N_features = 1`.

The protocol follows the QELM image classification of "Harnessing quantum extreme learning machines for image classification", Phys. Rev. Applied (2025) (`experiments/double_descent/double_descent.py`, config `conf/double_descent.yaml`):

- each image is reduced to `N` principal components (PCA fitted on the training images of the chosen digits), and each component to an angle `theta_i` in `[0, pi]`;
- qubit `i` starts in `R_y(theta_i)|0>`, a product state with no ancilla;
- one fixed quench `U = exp(-i H t)` acts on every image: a single TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j`, drawn once per seed, with `h = J` and `t = 10 / J`;
- the features are the one-site `<X_i>, <Y_i>, <Z_i>` and the correlators `<X_i X_j>, <Y_i Y_j>, <Z_i Z_j>` of the final state, plus a bias column: `3 N + 3 N (N - 1) / 2 + 1 = 109` columns at `N = 8`;
- the readout is the ridge regression of `conf/training/regression/ridge.yaml` with `training.alpha = 0` (`RIDGE_ALPHA`): no penalty, `W = Phi^+ Y`, the minimum-norm interpolation below the threshold. It learns the one-hot target of each image (`K` outputs for `K` digits) and predicts the digit of the largest output. It is fitted on `n_train = round(alpha * N_features)` images drawn from a pool, 20 times per `alpha`, and scored on the whole MNIST test split of the chosen digits.

`N_features` is the **rank** of the feature matrix of the pool, not its number of columns, so that `alpha = 1` is exactly the size where the square system `Phi W = y` becomes solvable. With `alpha = N_train / N_features`, `alpha < 1` is the over-parameterized side (more weights than images, the readout interpolates the training set) and `alpha > 1` the under-parameterized one (least squares).

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_double_descent_notebook.py` does for a small sweep). `N` is the number of qubits, one per principal component. `SEEDS` draws the reservoir (the couplings `J_ij`), the pool and the classical ELM layer. `CLASSES` lists the digits to tell apart (`"3 5"` by default, `"3 5 8"` for a harder task). `RIDGE_ALPHA` is the ridge penalty, 0 to expose the peak. `DATASET=digits` uses the 8x8 digits bundled with scikit-learn instead of MNIST, without download. The sweep goes to this notebook's own folder, `outputs/double_descent/<date_time>/`, and `RUN = False` reads back the latest sweep of this folder.

In [ ]:
N = os.environ.get("N", "8")
SEEDS = os.environ.get("SEEDS", "0 1 2 3 4")
CLASSES = os.environ.get("CLASSES", "3 5")  # the digits to tell apart
RIDGE_ALPHA = os.environ.get("RIDGE_ALPHA", "0.0")  # 0: no penalty
DATASET = os.environ.get("DATASET", "mnist")
N_POOL = os.environ.get("N_POOL", "2000")
N_TEST = os.environ.get("N_TEST", "null")  # null: the whole test split
N_REPEATS = os.environ.get("N_REPEATS", "20")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

One run per seed. The first run downloads MNIST to `data/mnist/` (about 11 MB, gitignored) and checks the SHA-256 of the four files.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "double_descent"

if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N": N,
        "SEEDS": SEEDS,
        "CLASSES": CLASSES,
        "RIDGE_ALPHA": RIDGE_ALPHA,
        "DATASET": DATASET,
        "N_POOL": N_POOL,
        "N_TEST": N_TEST,
        "N_REPEATS": N_REPEATS,
        "OUT_DIR": str(sweep_dir),
    }
    run_with_progress(
        ["bash", "experiments/double_descent/double_descent.sh"],
        total=len(SEEDS.split()),
        env=env,
        show=("N", "seed"),
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Accuracies and sanity checks

The readout fitted on the whole pool, scored on the test set, averaged over the seeds, next to the three checks of the issue:

1. **Classical limit.** The classical ELM has as many random ReLU neurons as the QELM has observables, with the same readout. The QELM should be comparable to it. `pixels` is the classical reference of the task, a linear readout on all 784 pixels fitted on every training image.
2. **t = 0.** `qelm_t0` measures the same observables on the encoded state, before any dynamics. Its features are checked against their closed form on the product state (`<P_i> = r_i^P`, `<A_i B_j> = r_i^A r_j^B` for the Bloch vectors `r_i`): the largest deviation is printed, 0 up to rounding. So the `t = 0` readout is exactly a linear regression on these fixed functions of the encoded input. Its rank is lower than the QELM's (`<Y_i> = 0` and `<Y_i Y_j> = 0` for a real state), which moves its threshold, and the sweep accounts for it by using the rank.
3. **Peak at alpha = 1.** For each model, the `alpha` where the median test MSE is largest, per seed and for the average over the seeds.

In [ ]:
import importlib.util

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_double_descent", "experiments/double_descent/plot_double_descent.py"
)
plot_double_descent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_double_descent)

runs = plot_double_descent.load_runs(sweep_dir)
table = plot_double_descent.summary(runs)
print(
    f"{table['n_runs']} seeds, digits {table['classes']}, "
    f"ridge alpha {table['ridge_alpha']}"
)
print(f"{'model':>8} {'rank':>10} {'columns':>8} {'accuracy':>17} {'peak alpha':>11}")
for model, row in table["models"].items():
    ranks = sorted(set(row["n_features"]))
    print(
        f"{model:>8} {'/'.join(map(str, ranks)):>10} {row['n_columns']:>8} "
        f"{row['accuracy_mean']:>8.4f} ± {row['accuracy_std']:<6.4f} "
        f"{row['peak_alpha_of_mean']:>11.3g}"
    )
if "accuracy_pixels_mean" in table:
    print(f"pixels: accuracy {table['accuracy_pixels_mean']:.4f}")
print(f"t = 0 features vs closed form: {table['t0_max_deviation']:.1e}")

## Double descent

Test and train errors against `alpha`, the median over the training sets of each seed for the MSE (its tails are heavy near the peak) and the mean for the classification error, then the mean over the seeds with the standard error. The dashed line is `alpha = 1`. Below it the train error is 0: the readout interpolates. The figure and its data are saved in the sweep folder, `double_descent.png` and `double_descent.json`, with `summary.json` and the confusion matrices below (`confusion_matrices.png` and `.json`).

In [ ]:
import matplotlib.pyplot as plt

table = plot_double_descent.plot_sweep(sweep_dir)
figure_path = sweep_dir / "double_descent.png"
print(f"figure: {figure_path}")

# Draw it again to show it inline (plot_sweep closes its figure).
fig, axes = plot_double_descent.draw(runs)
plt.show()

## Confusion matrices

For each model, the readout fitted on the whole pool, on the test set, summed over the seeds and normalized by row: entry `[i, j]` is the fraction of the test images of the true digit `i` predicted as digit `j`. A strong diagonal is an accurate readout; the off-diagonal entries show which digits it confuses, which matters most with more than two digits. The matrices (counts) are in each run's `.npz` as `confusion_<model>` and in `confusion_matrices.json`.

In [ ]:
fig_confusion, axes_confusion = plot_double_descent.draw_confusion(runs)
plt.show()

## State encodings (issue #196)

The sweep above encodes each image with one rotation per qubit. The same paper compares other ways of preparing the input state, all of them product states (`QELM.encoding.ProductEncoding`, one file each in `src/QELM/encoding/`). Qubit `i` encodes one or two consecutive features of `x = (x_1, ..., x_M)`, the principal components of the image mapped linearly onto the interval each encoding expects:

| Encoding | State of one qubit | Features per qubit | Interval |
| --- | --- | --- | --- |
| `angle_encoding` | `cos(x_1/2)|0> + sin(x_1/2)|1>` | 1 (`M = N`) | `[0, pi]` |
| `dense_angle_encoding` | `cos(x_1/2)|0> + e^{i x_2} sin(x_1/2)|1>` | 2 (`M = 2 N`) | `[0, pi]`, `[0, pi]` |
| `uniform_bloch_encoding` | `sqrt(x_1)|0> + e^{i x_2} sqrt(1 - x_1)|1>` | 2 | `[0, 1]`, `[0, pi]` |
| `general_encoding` | `(x_1|0> + x_2|1>) / sqrt(x_1^2 + x_2^2)` | 2 | `[0, 1]`, `[0, 1]` |

`angle_encoding` is the state of the sweep above (`rotation_encoding`, axis `y`), with the bounds of its input checked. The three others read twice as many principal components with the same `N` qubits, so the same reservoir and the same observables (the same `N_features`) see more of the image. An input outside its interval raises `OutOfBoundsError`.

First a small test of the encodings on a few images, then the comparison: the double descent above once per encoding, on the same seeds, digits and pool.

### Small test: the encodings on a few images

For each encoding: `M` principal components of `N_CHECK` training images of the chosen digits, mapped onto its interval as the experiment does (`encoding_inputs`), and their encoded states. The check prints, per encoding, the number of inputs, the largest deviation of the norms from 1, the largest deviation of the simulated `<X_i>, <Y_i>, <Z_i>` from their closed form on the product state (both 0 up to rounding), the mean of `|<Y_i>|` (0 for a real state: angle and general encodings), and whether an input pushed out of its interval raises.

In [ ]:
import numpy as np

from QELM.datasets import AngleEncoder, load_digit_classes
from QELM.encoding import OutOfBoundsError, get_encoding
from QELM.evaluation.double_descent import expectations, observable_operators

# The experiment script is not a package either: import it by path.
spec = importlib.util.spec_from_file_location(
    "double_descent_experiment", "experiments/double_descent/double_descent.py"
)
double_descent_experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(double_descent_experiment)

N_CHECK = int(os.environ.get("N_CHECK", "50"))  # images of the small test
n_qubits = int(N)
data = load_digit_classes(
    dataset=DATASET,
    classes=[int(d) for d in CLASSES.split()],
    root="data/mnist",
    download=True,
    seed=0,
)
_, bloch_ops = observable_operators(n_qubits, ["X", "Y", "Z"])

encoding_check = {}
print(f"{'encoding':>24} {'inputs':>6} {'norm':>9} {'Bloch':>9} {'|<Y>|':>7}  bounds")
for name in double_descent_experiment.ENCODINGS[1:]:
    encoding = get_encoding(name)({"N": n_qubits})
    bounds = double_descent_experiment.input_bounds(encoding)
    pca = AngleEncoder(len(bounds), angle_range=1.0).fit(data.x_train)
    x = double_descent_experiment.encoding_inputs(
        pca.transform(data.x_train[:N_CHECK]), bounds
    )
    psi = encoding.states(x)
    norm = np.max(np.abs(np.linalg.norm(psi, axis=1) - 1))
    simulated = expectations(psi, bloch_ops).reshape(len(x), 3, n_qubits)
    closed = encoding.bloch_vectors(x).transpose(0, 2, 1)
    bloch = np.max(np.abs(simulated - closed))
    try:  # a feature above its interval must raise
        encoding.states(x + (bounds[:, 1] - bounds[:, 0]))
        raises = False
    except OutOfBoundsError:
        raises = True
    encoding_check[name] = {
        "n_inputs": len(bounds),
        "norm_deviation": float(norm),
        "bloch_deviation": float(bloch),
        "mean_abs_Y": float(np.mean(np.abs(simulated[:, 1]))),
        "out_of_bounds_raises": raises,
    }
    print(
        f"{name:>24} {len(bounds):>6} {norm:>9.1e} {bloch:>9.1e} "
        f"{encoding_check[name]['mean_abs_Y']:>7.3f}  "
        f"{'raises' if raises else 'DOES NOT RAISE'}"
    )

### Comparison of the encodings

The sweep of the double descent above, once per encoding of `ENCODINGS`, with the same `N`, seeds, digits, pool and ratios: `outputs/double_descent_encodings/<date_time>/<encoding>/`, one sweep folder per encoding (with its own `double_descent.png`). The figure shows, for the QELM, the test MSE and the test error against `alpha`, one curve per encoding, and the table the accuracy of each model with each encoding. The figure and its data are `encodings.png` and `encodings.json` in the comparison folder, the table `encodings_summary.json`. `RUN = False` reads back the latest comparison.

In [ ]:
ENCODINGS = os.environ.get(
    "ENCODINGS",
    "angle_encoding dense_angle_encoding uniform_bloch_encoding general_encoding",
).split()
EXPERIMENT_ENCODINGS = "double_descent_encodings"

if RUN:
    encodings_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT_ENCODINGS)
    for name in ENCODINGS:
        print(f"encoding: {name}")
        run_with_progress(
            ["bash", "experiments/double_descent/double_descent.sh"],
            total=len(SEEDS.split()),
            env={**env, "ENCODING": name, "OUT_DIR": str(encodings_dir / name)},
            show=("N", "seed"),
        )
else:
    encodings_dir = latest_sweep(OUT_ROOT / EXPERIMENT_ENCODINGS)
print(f"comparison: {encodings_dir}")

encodings_table = plot_double_descent.plot_encodings(encodings_dir)
print(
    f"{'encoding':>24} {'inputs':>6} " + " ".join(f"{m:>16}" for m in table["models"])
)
for name, row in encodings_table.items():
    accuracies = " ".join(
        f"{r['accuracy_mean']:>7.4f} ± {r['accuracy_std']:<6.4f}"
        for r in row["models"].values()
    )
    print(f"{name:>24} {row['n_inputs']:>6} {accuracies}")

by_encoding = plot_double_descent.load_encodings(encodings_dir)
fig_encodings, axes_encodings = plot_double_descent.draw_encodings(by_encoding)
plt.show()